Import Libraries

In [1]:
pip install numpy pandas matplotlib seaborn scikit-learn scipy catboost xgboost lightgbm optuna joblib jupyter notebook ipykernel kaggle kagglehub plotly missingno tqdm

  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl (12.9 MB)
  Using cached pandas-2.3.3-cp310-cp310-win_amd64.whl (11.3 MB)
  Using cached matplotlib-3.10.9-cp310-cp310-win_amd64.whl (8.2 MB)
  Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
  Using cached scikit_learn-1.7.2-cp310-cp310-win_amd64.whl (8.9 MB)
  Using cached scipy-1.15.3-cp310-cp310-win_amd64.whl (41.3 MB)
  Using cached catboost-1.2.10-cp310-cp310-win_amd64.whl (100.2 MB)
  Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)
  Using cached lightgbm-4.7.0-py3-none-win_amd64.whl (1.4 MB)
  Using cached optuna-4.9.0-py3-none-any.whl (425 kB)
  Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
  Using cached jupyter-1.1.1-py2.py3-none-any.whl (2.7 kB)
  Using cached notebook-7.6.1-py3-none-any.whl (5.5 MB)
  Using cached kaggle-1.7.4.5-py3-none-any.whl (181 kB)
  Using cached kagglehub-1.0.2-py3-none-any.whl (70 kB)
  Using cached plotly-6.9.0-py3-none-any.whl (9.9 MB)
  Using cached tqdm-4.69.1-py3-non


[notice] A new release of pip available: 22.2.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier

# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [4]:
from src.preprocessing import fill_categorical_missing
from src.feature_engineering import create_features
from src.evaluation import evaluate_model

In [5]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print(train.shape)
print(test.shape)

(690088, 15)
(295753, 14)


In [6]:
train = create_features(train)
test = create_features(test)

print(train.shape)
print(test.shape)

(690088, 20)
(295753, 19)


In [7]:
new_features = [
    "activity_score",
    "calories_per_step",
    "water_per_exercise",
    "exercise_sleep_ratio",
    "bmi_category"
]

train[new_features].head()

,activity_score,calories_per_step,water_per_exercise,exercise_sleep_ratio,bmi_category
0,26254.8,1.638282,0.089423,3.183280,overweight
1,493560.9,0.198746,0.024754,7.641654,overweight
2,541629.6,0.189069,0.040921,6.057234,normal
3,429722.6,0.366551,0.033169,10.508772,normal
4,302864.0,0.388762,0.047872,5.589307,overweight


In [8]:
X = train.drop(columns=["health_condition"])
y = train["health_condition"]

X_test = test.copy()

In [9]:
X = fill_categorical_missing(X)
X_test = fill_categorical_missing(X_test)

In [10]:
cat_features = X.select_dtypes(include="object").columns.tolist()

print(cat_features)

['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender', 'bmi_category']


In [11]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [13]:
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function="MultiClass",
    eval_metric="Accuracy",
    random_seed=42,
    verbose=100
)
model

CatBoostClassifier(depth=8, eval_metric='Accuracy', iterations=1000, learning_rate=0.05, loss_function='MultiClass', random_seed=42, verbose=100)

In [14]:
model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_valid, y_valid),
    use_best_model=True
)

0:	learn: 0.9402866	test: 0.9425147	best: 0.9425147 (0)	total: 1.01s	remaining: 16m 51s
100:	learn: 0.9658485	test: 0.9661711	best: 0.9661711 (99)	total: 1m 22s	remaining: 12m 13s
200:	learn: 0.9664590	test: 0.9664174	best: 0.9664174 (200)	total: 2m 58s	remaining: 11m 48s
300:	learn: 0.9670259	test: 0.9665261	best: 0.9665551 (246)	total: 4m 37s	remaining: 10m 44s
400:	learn: 0.9677722	test: 0.9665478	best: 0.9666058 (385)	total: 6m 16s	remaining: 9m 21s
500:	learn: 0.9683319	test: 0.9666565	best: 0.9666855 (452)	total: 7m 50s	remaining: 7m 48s
600:	learn: 0.9689822	test: 0.9667507	best: 0.9668014 (596)	total: 9m 24s	remaining: 6m 15s
700:	learn: 0.9694966	test: 0.9668666	best: 0.9668956 (686)	total: 11m 2s	remaining: 4m 42s
800:	learn: 0.9700346	test: 0.9669101	best: 0.9669101 (789)	total: 12m 40s	remaining: 3m 8s
900:	learn: 0.9705689	test: 0.9668666	best: 0.9669826 (831)	total: 14m 14s	remaining: 1m 33s
999:	learn: 0.9710562	test: 0.9669971	best: 0.9669971 (999)	total: 15m 47s	remain

CatBoostClassifier(depth=8, eval_metric='Accuracy', iterations=1000, learning_rate=0.05, loss_function='MultiClass', random_seed=42, verbose=100)

Evaluate

In [15]:
pred = model.predict(X_valid)

evaluate_model(y_valid, pred)

Balanced Accuracy
0.8738720050994512

Classification Report
              precision    recall  f1-score   support

     at-risk       0.97      0.99      0.98    118512
         fit       0.94      0.83      0.88      7961
   unhealthy       0.95      0.80      0.87     11545

    accuracy                           0.97    138018
   macro avg       0.95      0.87      0.91    138018
weighted avg       0.97      0.97      0.97    138018

Confusion Matrix
[[117611    423    478]
 [  1362   6569     30]
 [  2247     15   9283]]


Feature Importance

In [16]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

importance.head(20)

,Feature,Importance
1,sleep_duration,27.925308
9,stress_level,25.674990
11,physical_activity_level,10.727754
3,bmi,8.314283
2,heart_rate,2.754001
7,water_intake,2.680963
14,activity_score,2.561779
0,id,2.478773
17,exercise_sleep_ratio,2.214662
4,calorie_expenditure,2.104915
